# מעבדה 05 — תגובת מתקף וקונבולוציה

במעבדה הזו תבעטו במתנד פעם אחת ותקבלו הכול: את תדירותו, את הריסון שלו, את מקדם הטיב שלו,
ואת עקומת התהודה השלמה שמודול 02 נזקק לסריקת תדרים שלמה כדי למדוד. אחר כך תבנו את התגובה
לכוחות שרירותיים מתוך בעיטות בלבד, ותשוו אותה לאינטגרטור שמעולם לא שמע על פונקציית גרין.

המדידה האחרונה היא כל עניינו של המודול: שתי דרכים שונות לחלוטין להגיע ל-$Q$ מרשומה רועשת
אחת, לכל אחת אי-ודאות משלה, ומבחן האם הן מסכימות.

עבדו לפי הסדר. היכן שהמחברת מבקשת מכם לנבא, רשמו את הניבוי שלכם בתא המיועד **לפני** שתריצו
את התא הבא.

## מפרט המודל

| | |
|---|---|
| **מערכת** | המתנד של מודול 02 הנצפה כהעתקה מהיסטוריית כוח להיסטוריית העתק |
| **דינמיקה** | $m\ddot{x} + b\dot{x} + kx = F(t)$, נפתרת כקונבולוציה $x = G * F$ או ב-velocity Verlet |
| **תנאי שפה** | במנוחה לפני שהכוח מתחיל — התנאי הסיבתי הבוחר את פונקציית גרין הזו |
| **אנסמבל** | זוג קלט–פלט דטרמיניסטי אחד לכל כוח; רעש המדידה גאוסי, בלתי תלוי ובעל זרע קבוע |
| **מוזנח** | המקור המיקרוסקופי של $b$, וכל אי-ליניאריות — סופרפוזיציה של בעיטות *היא* ליניאריות |
| **תקף כאשר** | התגובה ליניארית, הכוח חסום ומתחיל בזמן סופי, הדגימה הרבה מעל $\omega_d$ |
| **אופני כישלון** | קפיצים לא ליניאריים; תוכן כוח מעל נייקוויסט; דה-קונבולוציה נאיבית, המגבירה רעש ללא גבול |

כל הפיזיקה חיה ב-`wavelab.oscillators` וב-`wavelab.fourier` — פתחו אותם וקראו אותם.


In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import fourier, measurement, oscillators
from wavelab.validation import seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
DAMPING = 0.1  # kg/s  ->  Q = 20
OMEGA0 = oscillators.natural_frequency(MASS, STIFFNESS)
PERIOD = 2 * np.pi / OMEGA0
GAMMA = oscillators.damping_rate(MASS, DAMPING)
OMEGA_D = np.sqrt(OMEGA0**2 - GAMMA**2 / 4)
T_D = 2 * np.pi / OMEGA_D
TRUE_Q = oscillators.quality_factor(MASS, STIFFNESS, DAMPING)

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   omega_d = {OMEGA_D:.6f} rad/s   Q = {TRUE_Q}")
print(f"the ringing outlives the kick by 2/gamma = {2 / GAMMA:.1f} s"
      f" = {2 / GAMMA / PERIOD:.1f} periods")


## חלק 1 — בעיטה אחת

`oscillators.impulse_response` היא $G(t)$: התנועה שלאחר מתקף *יחידה*, ולא יותר מהדעיכה
החופשית של מודול 01 שהותחלה עם $x = 0$ ו-$v = 1/m$. הביטו במה שהיא עושה בראשית — היא מתחילה
באפס ויוצאת בשיפוע $1/m$ — ובמעטפת שבתוכה היא חיה.


In [ ]:
t = np.linspace(0.0, 12 * PERIOD, 3000)
green = oscillators.impulse_response(t, MASS, STIFFNESS, DAMPING)
envelope = np.exp(-GAMMA * t / 2) / (MASS * OMEGA_D)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(t, green, lw=1.2, label="G(t)")
ax.plot(t, envelope, ls="--", color="0.6", lw=0.9)
ax.plot(t, -envelope, ls="--", color="0.6", lw=0.9)
ax.axhline(0.0, color="0.85", lw=0.8)
ax.set_xlabel("t (s)")
ax.set_ylabel("G  (m per N s)")
ax.set_title("the impulse response, inside its decaying envelope")
plt.tight_layout()
plt.show()

slope = (green[1] - green[0]) / (t[1] - t[0])
print(f"G(0)          = {green[0]:.3e}      (a kick moves nothing instantly)")
print(f"slope at t=0  = {slope:.4f}     vs 1/m = {1 / MASS}")
print(f"peak of G     = {green.max():.4f}     vs 1/(m omega_d) = {1 / (MASS * OMEGA_D):.4f}")

# Read omega_d straight off the record the way an experimenter would, from the crossings.
measured_omega_d = measurement.frequency_from_zero_crossings(t, green)
print(f"omega_d from the crossings = {measured_omega_d:.4f}  vs {OMEGA_D:.4f}")


## חלק 2 — לנבא

לפני שתריצו את התא הבא, התחייבו לתשובות.

1. אתם עומדים כעת להתמיר את הדעיכה המצלצלת הבודדת הזו. כיצד אתם מצפים ש-$|\hat{G}(\omega)|$
   ייראה — קוץ, גבשושית, משהו אחר? באיזה רוחב?
2. מודול 02 מדד עקומת תהודה בסריקת הנעה על פני עשרות תדירויות, תוך המתנה לדעיכת תגובת
   המעבר בכל אחת. כמה קרוב יכולה בעיטה אחת להגיע לעקומה הזו?


**הניבוי שלכם:**

1.
2.


## חלק 3 — עקומת התהודה, מבעיטה אחת

התמירו את הדעיכה המצלצלת והניחו מעליה את התגובה הסרוקה של מודול 02. שום דבר מהסריקה לא שימש
ליצירת הדעיכה, ושום דבר מהדעיכה לא שימש ליצירת הסריקה.

פרט אחד חשוב ואינו קוסמטי. תחת הגרעין הישר של הקורס הזה $e^{-\ii\omega t}$, $\hat{G}$ נושאת
$+\ii\gamma\omega$ היכן שהפאזור $X$ של מודול 02 נושא $-\ii\gamma\omega$, ולכן השתיים הן
**צמודות מרוכבות**. התא מודד מה קורה אם שוכחים את זה.


In [ ]:
DT = 0.01
N = 2**15
centred = (np.arange(N) - N // 2) * DT  # the grid `fourier.spectrum` assumes: t = 0 in the middle
kick = oscillators.impulse_response(centred, MASS, STIFFNESS, DAMPING)
omega, kick_spectrum = fourier.spectrum(kick, DT)

band = (omega >= 0) & (omega < 3 * OMEGA0)
swept = oscillators.steady_state_response(omega[band], MASS, STIFFNESS, DAMPING, 1.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.4))
left.plot(omega[band], np.abs(kick_spectrum[band]), lw=2.4, color="0.75", label="from one kick")
left.plot(omega[band], np.abs(swept), lw=1.0, color="crimson", label="module 02 sweep")
left.set_xlabel("omega (rad/s)"), left.set_ylabel("|response|"), left.legend()
right.plot(omega[band], np.angle(kick_spectrum[band]), lw=2.4, color="0.75")
right.plot(omega[band], -np.angle(swept), lw=1.0, color="crimson")
right.set_xlabel("omega (rad/s)"), right.set_ylabel("phase of G-hat (rad)")
plt.tight_layout()
plt.show()

worst = np.max(np.abs(kick_spectrum[band] - np.conj(swept))) / np.max(np.abs(swept))
wrong = np.max(np.abs(kick_spectrum[band] - swept)) / np.max(np.abs(swept))
print(f"kick spectrum vs conjugated sweep: worst relative difference {worst:.2e}")
print(f"vs the sweep taken at face value:  {wrong:.2f}"
      "   <- forgetting the conjugate is a 200% error")


## חלק 4 — רכבות של בעיטות

שתי בעיטות, והמרווח קובע הכול. בפאזה הן מתחברות, באנטי-פאזה הן כמעט מתבטלות — כמעט, מפני
שהראשונה כבר דעכה מעט עד שהשנייה נוחתת. הצורות הסגורות הן $1 + e^{-\pi/Q}$ ו-
$1 - e^{-\pi/2Q}$.


In [ ]:
alone = oscillators.impulse_response(t, MASS, STIFFNESS, DAMPING)
settled = t > 3 * T_D
reference = np.max(np.abs(alone[settled]))

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(t, alone, lw=0.9, color="0.7", label="one kick")

print("  spacing              measured   predicted")
for name, spacing, predicted in (
    ("one period", T_D, 1 + np.exp(-np.pi / TRUE_Q)),
    ("half a period", T_D / 2, abs(1 - np.exp(-np.pi / (2 * TRUE_Q)))),
    ("a quarter period", T_D / 4, np.sqrt(2.0)),
):
    pair = alone + oscillators.impulse_response(t - spacing, MASS, STIFFNESS, DAMPING)
    late = t > spacing + 3 * T_D
    ratio = np.max(np.abs(pair[late])) / reference
    ax.plot(t, pair, lw=1.2, label=name)
    print(f"  {name:18s}   {ratio:6.3f}     {predicted:6.3f}")

ax.set_xlabel("t (s)"), ax.set_ylabel("x (m)"), ax.legend(fontsize=8)
ax.set_title("two kicks, three spacings")
plt.tight_layout()
plt.show()


## חלק 5 — כל כוח שהוא

`convolution_response` מחברת בעיטות. `simulate_forced` מקדמת את המשוואה הדיפרנציאלית בזמן
ואינה יודעת דבר על פונקציות גרין. אם התיאוריה נכונה הן חייבות להסכים על כל כוח שתעלו על
הדעת.


In [ ]:
dt = PERIOD / 400
n = 4000
grid = np.arange(n) * dt
forces = {
    "step": np.ones(n),
    "ramp": np.clip(grid / (4 * PERIOD), 0.0, 1.0),
    "burst": np.where(grid < 5 * PERIOD, np.cos(OMEGA0 * grid), 0.0),
    "noise": rng.normal(0.0, 1.0, n),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 5), sharex=True)
worst_gap = 0.0
for axis, (name, force) in zip(axes.ravel(), forces.items(), strict=True):
    convolved = oscillators.convolution_response(force, dt, MASS, STIFFNESS, DAMPING)
    stepped = oscillators.simulate_forced(MASS, STIFFNESS, 0.0, 0.0, dt, force, damping=DAMPING)
    gap = float(np.max(np.abs(convolved - stepped.positions)))
    worst_gap = max(worst_gap, gap / np.max(np.abs(convolved)))
    axis.plot(grid, convolved, lw=2.2, color="0.75")
    axis.plot(stepped.times, stepped.positions, lw=0.8, color="crimson")
    axis.set_title(f"{name}: max gap {gap:.1e} m", fontsize=9)
    axis.set_ylabel("x (m)")
    print(f"  {name:6s}: max |convolution - Verlet| = {gap:.2e} m"
          f"   (peak {np.max(np.abs(convolved)):.3f} m)")
axes[1, 0].set_xlabel("t (s)"), axes[1, 1].set_xlabel("t (s)")
plt.tight_layout()
plt.show()


## חלק 6 — כוח חסר תכונות, תשובה מצלצלת

לכוח המדרגה יש בדיוק תכונה אחת בכל ההיסטוריה שלו. צפו במה שהמתנד עושה איתו, וכיצד החריגה
תלויה ב-$Q$ בלבד — הצורה הסגורה היא $\exp(-\pi\gamma/2\omega_d)$.


In [ ]:
ts = np.linspace(0.0, 30 * PERIOD, 6000)
critical = 2 * np.sqrt(MASS * STIFFNESS)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.axhline(1.0, color="0.8", ls="--", lw=1.0)
print("     Q   overshoot   closed form")
for damping in (critical, 2.0, 1.0, 0.4, 0.1):
    q = oscillators.quality_factor(MASS, STIFFNESS, damping)
    response = oscillators.step_response(ts, MASS, STIFFNESS, damping, 1.0) * STIFFNESS
    gamma_d = oscillators.damping_rate(MASS, damping)
    squared = OMEGA0**2 - gamma_d**2 / 4
    predicted = np.exp(-np.pi * gamma_d / (2 * np.sqrt(squared))) if squared > 0 else 0.0
    ax.plot(ts, response, lw=1.3, label=f"Q = {q:.2f}")
    print(f"  {q:5.2f}   {100 * (response.max() - 1):6.2f}%     {100 * predicted:6.2f}%")

ax.set_xlabel("t (s)"), ax.set_ylabel("x / (F0/k)"), ax.legend(fontsize=8)
ax.set_title("step response: the force has one feature, the motion has dozens")
plt.tight_layout()
plt.show()


## חלק 7 — שתי דרכים ל-$Q$, עם סרגל שגיאה

זוהי המדידה שסביבה בנוי המודול. דעיכה מצלצלת רועשת אחת, נקראת בשתי דרכים:

- **המעטפת**, בתחום הזמן, עם `q_from_ringdown`;
- **רוחב הקו**, בתחום התדר, עם `q_from_linewidth`.

דבר אינו עובר ביניהן מלבד הנתונים. שמונה זרעים בלתי תלויים נותנים לכל אחת אי-ודאות, ואז
השאלה היא האם הן מסכימות בתוכה.


In [ ]:
NOISE = 2e-4 * float(np.max(np.abs(kick)))


def q_from_envelope(generator):
    """Time domain: how fast does the ringing die away?"""
    noisy = measurement.add_noise(kick, NOISE, generator)
    return oscillators.q_from_ringdown(centred[N // 2:], noisy[N // 2:])


def q_from_line(generator):
    """Frequency domain: how wide is the line? Fitted, not read off two crossings — see part 8."""
    noisy = measurement.add_noise(kick, NOISE, generator)
    w, spec = fourier.spectrum(noisy, DT)
    keep = (w > OMEGA0 - 4 * GAMMA) & (w < OMEGA0 + 4 * GAMMA)
    return oscillators.q_from_linewidth(w[keep], np.abs(spec[keep]))


ring = seed_study(q_from_envelope, n_seeds=8, base_seed=7)
line = seed_study(q_from_line, n_seeds=8, base_seed=7)

print(f"  Q from the envelope   = {ring.mean:.4f} +/- {ring.standard_error:.4f}")
print(f"  Q from the linewidth  = {line.mean:.4f} +/- {line.standard_error:.4f}")
print(f"  true Q                = {TRUE_Q}")
print(f"  the two routes differ by {abs(ring.mean - line.mean):.4f}, "
      f"or {100 * abs(ring.mean - line.mean) / TRUE_Q:.2f}% of Q")
print()
print("  Note how small the seed-to-seed scatter is compared with the offset from the true")
print("  value. Both routes carry a systematic larger than their noise, and telling those")
print("  two kinds of error apart is the whole skill.")


## חלק 8 — המלכודת

`q_from_bandwidth` מקבלת בדיוק את אותם ארגומנטים כמו `q_from_linewidth` ועונה על אותה שאלה
באינטרפולציה של שתי חציות חצי ההספק. על עקומה *סרוקה* זה הדבר הנכון לעשות. על הספקטרום של
דעיכה מצלצלת זו מלכודת, והיא לא תזהיר אתכם.

הסיבה היא חשבון, לא רעש. לרשומה באורך $T$ יש תאים במרווח $2\pi/T$, ולקו שעליה לפתור יש רוחב
$\gamma$. רשומה המחזיקה $n$ זמני דעיכה של המשרעת היא באורך $T = 2n/\gamma$, ולכן מספר התאים
על פני הרוחב המלא הוא $n/\pi$ — **בלתי תלוי ב-$Q$**.


In [ ]:
# A record holding three amplitude e-foldings: realistic, since after that the ringing has
# fallen to 5% of where it started and is heading for the noise floor.
short_n = 2 * int((3 * 2 / GAMMA) / DT)
short_t = (np.arange(short_n) - short_n // 2) * DT
short_kick = oscillators.impulse_response(short_t, MASS, STIFFNESS, DAMPING)
w_s, spec_s = fourier.spectrum(short_kick, DT)
keep_s = (w_s > 0.5 * OMEGA0) & (w_s < 1.5 * OMEGA0)

bins = GAMMA / (w_s[1] - w_s[0])
crossings = oscillators.q_from_bandwidth(w_s[keep_s], np.abs(spec_s[keep_s]))
fitted = oscillators.q_from_linewidth(w_s[keep_s], np.abs(spec_s[keep_s]))

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(w_s[keep_s], np.abs(spec_s[keep_s]), marker="o", ms=3, lw=0.8)
ax.set_xlabel("omega (rad/s)"), ax.set_ylabel("|G-hat|")
ax.set_title(f"the line, as actually sampled: {bins:.2f} bins across its full width")
plt.tight_layout()
plt.show()

print(f"  bins across the full width : {bins:.2f}")
print(f"  q_from_bandwidth (crossings): {crossings:6.2f}"
      f"   <- {100 * (crossings / TRUE_Q - 1):+.0f}%")
print(f"  q_from_linewidth (fitted)   : {fitted:6.2f}   <- {100 * (fitted / TRUE_Q - 1):+.0f}%")
print(f"  true Q                      : {TRUE_Q:6.2f}")
print()
for e_folds in (3, 5, 10, 31):
    print(f"  {e_folds:2d} e-foldings -> {e_folds / np.pi:5.2f} bins across the width, "
          f"signal down to {np.exp(-e_folds):.1e} of its start")
print()
print("  Recording for longer is not the escape it looks like.")


## חלק 9 — לבטל קונבולוציה, ולהיכשל

אם $\hat{x} = \hat{G}\hat{F}$ אז $\hat{F} = \hat{x}/\hat{G}$, ולכן תנועה מדודה אמורה להחזיר
את הכוח שגרם לה. נסו זאת, והוסיפו מעט רעש.


In [ ]:
probe = np.where((centred > 0) & (centred < 6.0), 1.0, 0.0)  # a force we will try to recover
motion = fourier.convolve(kick, probe, DT)
_, green_spectrum = fourier.spectrum(kick, DT)
trusted = np.abs(green_spectrum) > 0.05 * np.max(np.abs(green_spectrum))

print("     noise      naive peak   truncated peak   (the true force peaks at 1.0)")
for level in (0.0, 1e-6, 1e-4, 1e-2):
    noisy_motion = motion + rng.normal(0.0, level * np.max(np.abs(motion)), N)
    _, noisy_spectrum = fourier.spectrum(noisy_motion, DT)
    naive = np.real(fourier.inverse_spectrum(noisy_spectrum / green_spectrum, DT))
    rescued = np.real(
        fourier.inverse_spectrum(np.where(trusted, noisy_spectrum / green_spectrum, 0.0), DT)
    )
    print(f"  {level:8.0e}   {np.max(np.abs(naive)):10.2e}   {np.max(np.abs(rescued)):10.3f}")

print()
print("  Dividing by G-hat multiplies by a large number wherever G-hat is small, and what")
print("  lives out there is noise. Refusing to divide below a threshold keeps the answer")
print("  bounded and throws away real information with the noise. Module 53 does better.")


In [ ]:
# 1. The kick spectrum reproduces module 02's swept response, once conjugated.
assert worst < 1e-4

# 2. And the conjugate is load-bearing: getting it wrong is not a small error.
assert wrong > 0.5

# 3. G starts at zero and leaves the origin at 1/m.
assert green[0] == 0.0
assert abs(slope - 1 / MASS) / (1 / MASS) < 1e-3

# 4. Causality: nothing moves before the kick.
before = oscillators.impulse_response(np.linspace(-5, -0.01, 50), MASS, STIFFNESS, DAMPING)
assert np.all(before == 0.0)

# 5. Convolution and Verlet agree on every force tried, to well under a percent of the peak.
assert worst_gap < 1e-2

# 6. Two routes to Q from one record, agreeing to better than 1%.
assert abs(ring.mean - line.mean) / TRUE_Q < 1e-2
assert abs(ring.mean - TRUE_Q) / TRUE_Q < 2e-2
assert abs(line.mean - TRUE_Q) / TRUE_Q < 2e-2

# 7. The trap is real: on an under-resolved line the crossing route reads low and the fit does not.
assert bins < 2.0
assert abs(crossings - TRUE_Q) / TRUE_Q > 0.15
assert abs(fitted - TRUE_Q) / TRUE_Q < 0.06

print("all checks passed")
print(f"one kick reproduced the swept resonance curve to {worst:.1e} relative")


## חלק 10 — לחקור בעצמכם

הניחו בעיטות משלכם וצפו בתגובה מרכיבה את עצמה מתוכן. נסו שתי בעיטות באותו מרווח שמדדתם
בחלק 4; נסו מטח של עשר בזמנים אקראיים; נסו להעלות את $Q$ עד שהצלצול של כל בעיטה שורד עד
לפגישה עם הבאה.


In [ ]:
import ipywidgets as widgets


def explore(quality=20.0, n_kicks=4, spacing=1.0, seed=3):
    damping = np.sqrt(MASS * STIFFNESS) / quality
    gamma = oscillators.damping_rate(MASS, damping)
    omega_d = np.sqrt(OMEGA0**2 - gamma**2 / 4)
    period_d = 2 * np.pi / omega_d

    local = np.random.default_rng(seed)
    grid = np.linspace(0.0, max(14 * PERIOD, (n_kicks + 4) * spacing * period_d), 3000)
    times = np.arange(n_kicks) * spacing * period_d + 0.5 * PERIOD
    strengths = local.choice([-1.0, 1.0], n_kicks)

    parts = [s * oscillators.impulse_response(grid - t0, MASS, STIFFNESS, damping)
             for t0, s in zip(times, strengths, strict=True)]
    total = np.sum(parts, axis=0)

    fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 4), sharex=True,
                                      gridspec_kw={"height_ratios": [1, 2.4]})
    top.vlines(times, 0, strengths, color="darkorange", lw=1.6)
    top.axhline(0, color="0.85", lw=0.8), top.set_ylabel("F"), top.set_ylim(-1.3, 1.3)
    for part in parts:
        bottom.plot(grid, part, lw=0.8, color="0.75")
    bottom.plot(grid, total, lw=1.8)
    bottom.axhline(0, color="0.85", lw=0.8)
    bottom.set_xlabel("t (s)"), bottom.set_ylabel("x (m)")
    plt.tight_layout()
    plt.show()

    print(f"Q = {quality:.1f}   ringing lifetime 2/gamma = {2 / gamma:.1f} s = "
          f"{2 / gamma / period_d:.1f} periods   peak |x| = {np.max(np.abs(total)):.3f} m")


widgets.interact_manual(
    explore,
    quality=widgets.FloatSlider(min=1.0, max=40.0, step=0.5, value=20.0, description="Q"),
    n_kicks=widgets.IntSlider(min=1, max=8, step=1, value=4, description="kicks"),
    spacing=widgets.FloatSlider(min=0.25, max=3.0, step=0.05, value=1.0, description="spacing/T"),
    seed=widgets.IntSlider(min=0, max=20, step=1, value=3, description="seed"),
);


## בחנו את הבנתכם


In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "05-impulse-response.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")


## לפני שאתם הולכים

כתבו כמה משפטים על כל אחת, בתא שלמטה.

1. בחלק 3 שחזרתם עקומת תהודה שלמה מבעיטה אחת. מה *לא* קיבלתם שסריקת תדרים אמיתית הייתה
   נותנת לכם? חשבו על רעש, ועל מה שסריקה עושה ורשומה בודדת אינה יכולה.
2. בחלק 8 שני האומדנים נחלקו ב-18% על נתונים זהים, ואף אחד מהם לא זרק שגיאה. מה היה צריך
   להשתנות ב-`q_from_bandwidth` כדי שתסרב לעבודה, ומדוע מחבר ספרייה עשוי להחליט בסבירות
   שלא לגרום לה לסרב?
3. הדה-קונבולוציה של חלק 9 נכשלה ברעש של 1%. נניח שיכולתם לתכנן את כוח הקלט בעצמכם במקום
   לקבל אותו. איזה כוח היה מקל על השחזור ביותר, ומה אומרת תשובתכם על הסיבה שמהנדסים
   משתמשים בפטישים?


**התשובות שלכם:**

1.
2.
3.
